So, essentially in RAG we decided what the context (Knowledge Base) is. We decide what question is to be searched in the KB. The LLM:
- took the query and searched in KB (Human/Application Driven)
- It generated a response based on the results & the instructions provided. (LLM Driven)

The LLM just pieced everything together and gave the final answer.

With Agentic RAG (Function Calling) - We still decide the KB. But this time we just tell the LLM that it has a tool to search the KB, how to search and when to search. The LLM:
- took the query, evaluated whether to call the search tool and decided what args to pass (Agent Driven)
- searched in KB (Human/Application Driven), but used its own args (Agent)
- It generated a response based on the results & the instructions provided. (LLM Driven)

The LLM decided whether to use the tool, what arguments to give it, and whether another tool call was necessary; the application still executes the tool, while the LLM synthesizes the final answer.

## Normal LLM v/s RAGBase

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
MODEL="gemini-3.6-flash"

In [10]:
from ingest import load_faq_data, build_index
from rag_helper_gemini import RAGBase
from google import genai

documents = load_faq_data()
index = build_index(documents)

client = genai.Client() # Picks API_KEY from .env

assistant = RAGBase(
    index=index,
    llm_client=client,
    model=MODEL
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join! However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.


In [11]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

I don't know.


In [12]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. **Install Ollama** for your operating system from [https://ollama.com/download](https://ollama.com/download):
   - **macOS**: Download the `.pkg` and install it.
   - **Windows**: Download the `.msi` and install it.
   - **Linux**: Run `curl -fsSL https://ollama.com/install.sh | sh` in your terminal.

2. **Run the model** by opening a terminal and typing:
   ```bash
   ollama run llama3
   ```
   This will download the model (e.g., LLaMA 3), start it locally, and open an interactive chat interface.

You can test that the local server is running by executing:
```bash
curl http://localhost:11434
```


In [13]:
from google import genai

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="I just discovered the course. Can I join now?"
)

print(interaction.output_text)

I would love to help you, but as an AI assistant, I don't know which specific course or platform you are looking at! 

To help me give you the right information, could you share:
1. **The name of the course?**
2. **The website, instructor, or platform hosting it** (e.g., Coursera, Udemy, a specific university, or an independent website)?

---

### General rules of thumb while you check:
* **Self-Paced Courses (Udemy, Coursera, edX, Skillshare):** Yes! You can almost always enroll and start immediately, anytime.
* **Cohort-Based or Live Bootcamps:** It depends on whether registration is still open or if the current session has already started. Late enrollment is sometimes allowed during the first week.
* **University / Academic Courses:** You will need to check the official academic calendar or contact the instructor/registrar to see if the add/drop deadline has passed.

If you are trying to reach a specific instructor or support team, look for a **"Contact Us," "FAQ,"** or **"Support"*

In [14]:
from google import genai

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="How do I run Olama locally?"
)

print(interaction.output_text)

Running **Ollama** locally is very straightforward. Ollama allows you to run open-source large language models (like Llama 3, Mistral, and Phi-3) directly on your hardware.

Here is the step-by-step guide to set it up:

---

### Step 1: Install Ollama

Choose the instructions for your operating system:

*   **macOS:**
    *   Download the official installer from [ollama.com/download](https://ollama.com/download) and move it to your Applications folder.
    *   *Or using Homebrew:* Run `brew install ollama` in your terminal.
*   **Windows:**
    *   Download the `OllamaSetup.exe` installer from [ollama.com/download](https://ollama.com/download) and run it.
*   **Linux:**
    *   Open your terminal and run this single command:
        ```bash
        curl -fsSL https://ollama.com/install.sh | sh
        ```

---

### Step 2: Run Your First Model

Once installed, Ollama runs in the background. You interact with it using your command line (Terminal on Mac/Linux, or PowerShell/Command Promp

## Function/Tool Calling
Stateless - We have to send the whole history because LLMs are stateless between API calls. The memory is the list you send as input. If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. That means the question, the decision to call search, and the result we got back.

In [1]:
from ingest import load_faq_data, build_index
from google import genai
import json

documents = load_faq_data()
index = build_index(documents)

In [2]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [3]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [4]:
client = genai.Client()

history = [
    {
        "type": "user_input",
        "content": [{"type": "text", "text": "I just discovered the course. Can I join now??"}]
    }
]

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    store=False, # Stateless
    input=history,
    tools=[search_tool],
)

In [5]:
interaction # Very Important to understand this

Interaction(status='requires_action', model='gemini-3.6-flash', agent=None, id='', created='2026-09-13T10:37:11Z', updated='2026-09-13T10:37:11Z', system_instruction=None, tools=None, errors=None, usage=Usage(cached_tokens_by_modality=None, grounding_tool_count=None, input_tokens_by_modality=[ModalityTokens(modality='text', tokens=74)], output_tokens_by_modality=None, tool_use_tokens_by_modality=None, total_cached_tokens=0, total_input_tokens=74, total_output_tokens=22, total_thought_tokens=43, total_tokens=139, total_tool_use_tokens=0, raw_prompt_token=114, model_invocation_token_counts=[{'prompt_tokens_details': [{'modality': 'text', 'tokens': 114}], 'candidates_tokens_details': [{'modality': 'text', 'tokens': 28}], 'thoughts_tokens_details': [{'modality': 'text', 'tokens': 43}]}]), response_modalities=None, response_mime_type=None, previous_interaction_id=None, environment_id=None, service_tier='standard', webhook_config=None, steps=[ThoughtStep(signature='Ep8CCpwCARFNMg9Gu614VmBOej

In [6]:
functions = {
    "search": search
}

# Thought & Function-call steps
for step in interaction.steps:
    history.append(step.model_dump())
    if step.type == "function_call":
        result_of_function_call = functions[step.name](**step.arguments) # search(query)
        fn_result = {
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": [{"type": "text", "text": json.dumps(result_of_function_call)}],
            }
        history.append(fn_result)

In [7]:
print(history)

[{'type': 'user_input', 'content': [{'type': 'text', 'text': 'I just discovered the course. Can I join now??'}]}, {'signature': 'Ep8CCpwCARFNMg9Gu614VmBOejQ0hEzxKYhnvxmrFJdW2YUbwrjMxSLY0AtiYQBDChefRjRBeI/s+BahBjukMTJQM293qbo/bUP9cX4n4a98FW9O7bZDDJPCzPlQbAZ22+gbKzZvKnY48Ertmp6vEy8pAFa+TpIvWB2xNSIHl7NpC2cVwP9T08kF3ev1VyZlz8Q4gBDAjKqgrAdmrDH3ANdqBg6DI+HrKRpQ9j6BmABflu7+oqckAt+3Am+OAR/JAEbIjVJN0Sn3lTwb9Itt1HckO4uk6fy0okeIyzi8Bk34ddY7Ypu9YSiy5BTZmBV9x99+Gql6ZTrdVepWphTjQSTXTpk7Uwi9j3/+HcnHejtyIPIdPEl+jmfsVqkIZhqAFUc=', 'type': 'thought'}, {'arguments': {'query': 'join late late enrollment join now course start date'}, 'id': 'call_4084048', 'name': 'search', 'type': 'function_call'}, {'type': 'function_result', 'name': 'search', 'call_id': 'call_4084048', 'result': [{'type': 'text', 'text': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a 

In [9]:
interaction = client.interactions.create(
        model="gemini-3.6-flash",
        store=False,
        input=history,
        tools=[search_tool],
    )

In [10]:
interaction.output_text

'Yes, you can still join! \n\nYou can start whenever you want—the course videos and GitHub materials are publicly available, and registration is not strictly checked against an enrollment list. \n\nHowever, if you want to receive a course certificate, you must submit your project while project submissions are still open.'